# Fine-tune AgyssyyGPT on a free Kaggle GPU

This notebook runs **QLoRA** fine-tuning on your own `agys --log` conversations and
pushes the adapter to the Hugging Face Hub.

### Before you start
1. **Settings → Accelerator →** pick a GPU (T4 x2 or P100).
2. **Add-ons → Secrets →** add `HF_TOKEN` with your Hugging Face **write** token
   (https://huggingface.co/settings/tokens) and toggle it on for this notebook.
3. Put your data where this notebook can read it:
   - upload `dataset.jsonl` (from `agys --log` + `prepare_dataset.py`) as a Kaggle
     dataset, **or**
   - push it to the Hub in step 2 below and use `YOUR-NAME/agysyygpt-chats`.

> Free Kaggle GPU is roughly 30 hours/week. A 3B model on a few thousand examples
> trains in well under an hour.

In [ ]:
!git clone https://github.com/agyssyy/agysyygpt.git
%cd agysyygpt
!pip install -q -r training/requirements.txt

In [ ]:
import os
from huggingface_hub import login

login(token=os.environ["HF_TOKEN"])

## Configure the run

Set `DATASET` to either a local `dataset.jsonl` path or a Hub dataset id. Set
`HF_REPO` to where the trained adapter should be published.

In [ ]:
DATASET = "dataset.jsonl"                      # or "your-name/agysyygpt-chats"
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT = "outputs/agysyygpt-lora"
HF_REPO = "your-name/agysyygpt-lora"           # where to push the adapter
EPOCHS = 3
MAX_SEQ_LEN = 2048

In [ ]:
!python training/finetune_lora.py \
  --dataset {DATASET} \
  --base-model {BASE_MODEL} \
  --output {OUTPUT} \
  --epochs {EPOCHS} \
  --max-seq-len {MAX_SEQ_LEN} \
  --push-to-hub {HF_REPO}

## Next: run it locally with Ollama

Merge the adapter and export to GGUF (see `training/README.md`, step 4), then:

```sh
agys -p ollama -m agysyygpt
```